# ISU Campus Assistant — Gemini vs Ollama

This notebook asks **Google Gemini** (from `ISU-CampusAssistant-RAG-Demo.ipynb`) and **Gemma 4 12B on
Ollama** (from `ISU-CampusAssistant-RAG-Ollama.ipynb`) the same questions, then scores them side by side.

**Only the model changes.** Both get the same 50 Arm 1 questions, the same saved catalog data, the same
retrieved chunks, the same instructions and the same temperature. The code comes straight from the two
programs (the `reggiebot` package and the Gemini notebook) rather than being copied, so this always
compares them exactly as they are now.

| Section | What happens |
|---|---|
| **1. Settings** | How many questions, and how long to pause between Gemini calls |
| **2. Load** | Builds the search index once and sets up both models |
| **3. Ask** | Both models answer every question. Answers are saved as they arrive, so a re-run only asks what is missing |
| **4. Automatic scores** | Answered, cites the right source, links websites, declines trap questions, speed |
| **5. Blind grading** | You grade answers labelled **A** and **B** without knowing which model wrote which |
| **6. Verdict** | Which one is better, with the numbers behind it, saved to `experiments/compare/` |

### Before you run it

1. Keep this notebook in the same folder as the other two.
2. The saved data must exist. If you have run the crawler in either notebook, it does (`isu_catalog_cache`).
3. Ollama must be running with `gemma4:12b` downloaded, and `GOOGLE_API_KEY` must be in `.env`.
4. Shut down the other two notebooks' kernels first. All three share `isu_catalog_cache`.
5. `Run All`. A full run takes about 10 minutes, mostly the pauses between Gemini calls.

> **Gemini quota:** a full run uses about 50 Gemini requests. If the free tier runs out partway, the
> notebook carries on with Ollama alone, and running section 3 again later asks only the missing Gemini
> answers. For a quick trial first, set `QUESTION_LIMIT = 5`.

---
## 1 · Settings

In [1]:
from pathlib import Path

GEMINI_NOTEBOOK = Path("ISU-CampusAssistant-RAG-Demo.ipynb")
OLLAMA_NOTEBOOK = Path("ISU-CampusAssistant-RAG-Ollama.ipynb")

QUESTION_LIMIT = None   # None = all Arm 1 questions; a number = only the first N, for a quick trial
GEMINI_PAUSE   = 6      # seconds between Gemini calls - inside the free tier's per-minute limit
SKIP_PROBES    = True   # skip Gemini's startup test calls (saves 3 requests of quota per run)
BLIND_SEED     = 7      # decides which model is shown as "A" for each question while grading

---
## 2 · Load both programs

The Ollama program is the `reggiebot` package, so this cell imports it. Settings, data loading and the search
index all come from the package, and the Gemini side uses the same ones. The only part that differs is the
cell that writes answers: Gemini's is read from `ISU-CampusAssistant-RAG-Demo.ipynb` and run in its own private
namespace, next to `reggiebot.answer` for Ollama. The question set and scoring helpers come from the Ollama
notebook's §13. If either notebook is restructured, this cell stops with a message saying which cell it could
not find.

Nothing here crawls or launches the chat app: it only loads the saved data.

In [ ]:
import json

# The Ollama program is the reggiebot package: settings, data loading and search come from it, and the
# Gemini notebook uses the same ones. Only the cell that writes answers differs between the two.
from reggiebot.config import *
from reggiebot.collect import *
from reggiebot.index import *
from reggiebot.core import CatalogApp
from reggiebot import answer as ollama_answer

GEMINI_CONFIG_CELL = "import os\nimport sys\nfrom pathlib import Path"   # GEMINI_MODEL, PROBE_MODELS; reads .env
GEMINI_ANSWER_CELL = 'SYSTEM_PROMPT = """'
EVAL_CELLS = ["# The Arm 1 question set.", "import contextlib"]          # questions + scoring helpers (§13)


def code_cells(path):
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - keep this notebook in the same folder as the other two.")
    nb = json.loads(path.read_text(encoding="utf-8"))
    return ["".join(cell["source"]) for cell in nb["cells"] if cell["cell_type"] == "code"]


def cell_starting(cells, start, path):
    found = [src for src in cells if src.startswith(start)]
    if len(found) != 1:
        raise RuntimeError(f"{path.name}: expected 1 code cell starting with {start.splitlines()[0]!r}, "
                           f"found {len(found)}. Was that notebook restructured?")
    return found[0]


gemini_cells, ollama_cells = code_cells(GEMINI_NOTEBOOK), code_cells(OLLAMA_NOTEBOOK)
for start in EVAL_CELLS:
    exec(cell_starting(ollama_cells, start, OLLAMA_NOTEBOOK))

# Gemini's settings and answer cell run in a private copy of these definitions, so its Answerer,
# SYSTEM_PROMPT and helpers cannot collide with the Ollama ones.
GEMINI_NS = dict(globals())
exec(cell_starting(gemini_cells, GEMINI_CONFIG_CELL, GEMINI_NOTEBOOK), GEMINI_NS)
exec(cell_starting(gemini_cells, GEMINI_ANSWER_CELL, GEMINI_NOTEBOOK), GEMINI_NS)
GEMINI_NS["PROBE_MODELS"] = not SKIP_PROBES

APP = CatalogApp()
GEMINI = GEMINI_NS["Answerer"](APP.index)
OLLAMA = ollama_answer.Answerer(APP.index)
APP.answerer = OLLAMA                     # so load() does not build a third answerer
APP.load(progress=lambda msg: print("  ", msg))

QUESTIONS = list(ARM1_QUESTIONS[:QUESTION_LIMIT] if QUESTION_LIMIT else ARM1_QUESTIONS)
print()
for label, answerer in (("Gemini", GEMINI), ("Ollama", OLLAMA)):
    print(f"{label}: " + (f"ready - {answerer.model}" if answerer.available else f"NOT READY - {answerer.error}"))
print(f"{len(QUESTIONS)} questions to compare")

---
## 3 · Ask both models every question

For each question, Gemini answers first, then Ollama. Each model's answer goes through that notebook's own
`answer()`, so it searches, builds the prompt, retries and falls back exactly as the chat app does.

Answers are appended to `experiments/compare/answers.jsonl` as they arrive. Running this cell again skips
every question a model has already answered and retries only the missing ones — the quota fallbacks, for
example. If a model fails three times in a row (quota used up, Ollama not running), it is skipped for the
rest of the run and the other model carries on. Tick `START_OVER` to set the saved answers aside and ask
everything fresh; the old file is renamed, not deleted.

In [ ]:
import contextlib
import csv
import io
import random
import statistics

COMPARE_DIR  = CACHE_DIR / "experiments" / "compare"
ANSWERS_PATH = COMPARE_DIR / "answers.jsonl"     # one line per (model, question); re-runs append
GRADES_PATH  = COMPARE_DIR / "grades.csv"
MODELS       = {"gemini": GEMINI, "ollama": OLLAMA}
LABELS       = {"gemini": "Gemini", "ollama": "Ollama"}
STOP_ON      = ("quota", "offline", "missing", "not set up")   # three in a row and that model sits out

# How each notebook's non-answers begin (matches_only() in each answer cell).
FALLBACK_STARTS = {
    "**The model is busy": "busy",
    "**Gemini quota": "quota",
    "**Gemini rejected": "rejected",
    "**Ollama is not running": "offline",
    "**The model is not downloaded": "missing",
    "**Ollama rejected": "rejected",
    "**Closest matches": "not set up",
    "Nothing in the catalog matched": "no matches",
}


def status_of(text):
    for start, label in FALLBACK_STARTS.items():
        if text.startswith(start):
            return label
    return "answered" if text.strip() else "empty"


def load_answers():
    """Saved answers keyed by (model, question id); a later line replaces an earlier one."""
    saved = {}
    if ANSWERS_PATH.exists():
        for line in ANSWERS_PATH.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                saved[(rec["generator"], rec["id"])] = rec
    return saved


def ask_one(name, answerer, q):
    """One model, one question, scored with the same automatic checks Arm 1 uses."""
    trap = bool(q.get("trap"))
    valid = trap or any(is_relevant(q, doc) for doc in APP.index.documents)
    log = io.StringIO()
    started = time.time()
    with contextlib.redirect_stdout(log):              # keep retry chatter out of the progress lines
        reply = answerer.answer(q["question"], k=TOP_K)
    if isinstance(reply, tuple):       # the Gemini notebook's answerer still returns (text, hits)
        text, hits = reply
        status = status_of(text)
        retries = sum(1 for line in log.getvalue().splitlines() if " attempt " in line)
    else:                              # reggiebot's Answer carries its own status and retry count
        text, hits, status, retries = reply.text, reply.hits, reply.status, reply.retries
    scored = status == "answered" and valid
    return {
        "generator": name, "model": answerer.model or "", "id": q["id"], "category": q["category"],
        "question": q["question"], "rule": describe_rule(q), "status": status,
        "latency_s": round(time.time() - started, 2),
        "retries": retries,
        "context_ids": [hit["doc"]["id"] for hit in hits],
        "has_link": "](http" in text,
        "cites_expected": cites_expected(q, text, hits) if (scored and not trap) else None,
        "declined": declined(text) if (scored and trap) else None,
        "words": len(text.split()),
        "answer": text,
        "asked_at": datetime.now().isoformat(timespec="seconds"),
    }


def run_compare(questions):
    COMPARE_DIR.mkdir(parents=True, exist_ok=True)
    saved = load_answers()
    asking = {name: answerer.available for name, answerer in MODELS.items()}
    for name, answerer in MODELS.items():
        if not answerer.available:
            print(f"{LABELS[name]} is not ready, so it is skipped this run: {answerer.error}\n")
    streak = {name: 0 for name in MODELS}

    for n, q in enumerate(questions, 1):
        notes = []
        for name, answerer in MODELS.items():
            done = saved.get((name, q["id"]))
            if done and done["status"] == "answered" and done["model"] == (answerer.model or done["model"]):
                notes.append(f"{name} saved")
                continue
            if not asking[name]:
                notes.append(f"{name} -")
                continue
            rec = ask_one(name, answerer, q)
            with open(ANSWERS_PATH, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            saved[(name, q["id"])] = rec
            notes.append(f"{name} {rec['status']} {rec['latency_s']:.1f}s")
            streak[name] = streak[name] + 1 if rec["status"] in STOP_ON else 0
            if streak[name] >= 3:
                asking[name] = False
                print(f"   {LABELS[name]} failed 3 times in a row ({rec['status']}) - skipping it for the rest "
                      "of this run. Fix that and run this cell again; saved answers are kept.")
            elif name == "gemini" and n < len(questions):
                time.sleep(GEMINI_PAUSE)
        print(f"{q['id']:<4} " + "  ".join(f"{note:<24}" for note in notes) + q["question"][:46])

    missing = [(LABELS[name], q["id"]) for q in questions for name in MODELS
               if saved.get((name, q["id"]), {}).get("status") != "answered"]
    print(f"\nsaved to {ANSWERS_PATH}" +
          (f" - {len(missing)} answer(s) still missing; run this cell again to fill them in" if missing else
           " - both models answered every question"))
    return saved

In [4]:
RUN_COMPARE = True   #@param {type:"boolean"}
START_OVER  = False  #@param {type:"boolean"}

if not RUN_COMPARE:
    print("Skipped - tick RUN_COMPARE and run this cell to ask both models the questions.")
else:
    if START_OVER and ANSWERS_PATH.exists():
        old = ANSWERS_PATH.with_name(f"answers_{datetime.now():%Y%m%d_%H%M%S}.jsonl")
        ANSWERS_PATH.rename(old)
        print(f"previous answers set aside as {old.name}\n")
    run_compare(QUESTIONS)

C01  gemini saved              ollama saved            What are the prerequisites for IT 214?
C02  gemini saved              ollama saved            What is IT 140 about?
C03  gemini saved              ollama saved            How many credit hours is IT 150?
C04  gemini saved              ollama saved            What's the difference between IT 354 and IT 45
C05  gemini saved              ollama saved            Describe TEC 283.
C06  gemini saved              ollama saved            What does CTK303 cover?
C07  gemini saved              ollama saved            what is com 110
C08  gemini saved              ollama saved            Tell me about ENG 101.
C09  gemini saved              ollama saved            What is IT-344 about?
C10  gemini saved              ollama saved            Is MAT 120 an undergraduate or graduate course
T01  gemini saved              ollama saved            Which courses teach web application developmen
T02  gemini saved              ollama saved            Ar

---
## 4 · Automatic scores

These are the same automatic checks as Arm 1, run on both models:

| Measure | What it checks |
|---|---|
| **Answered** | The model wrote an answer instead of a fallback (quota, busy, Ollama not running) |
| **Cites the right source** | The answer names the expected course code, or links the expected website |
| **Website answers with a link** | Billing, housing and dining answers include a source link |
| **Trap questions declined** | For questions with no answer in the data, the reply says so instead of inventing one |
| **Seconds per answer** | Median time for one answer, retries included, pauses between questions not |

They are text matches, so they catch a missing source but not a wrong fact. Section 5 covers that.

In [5]:
def pct(num, den):
    return f"{num}/{den} ({num / den:.0%})" if den else "n/a"


def auto_scores(saved, questions):
    table = {}
    for name in MODELS:
        recs = [saved[(name, q["id"])] for q in questions if (name, q["id"]) in saved]
        ok = [r for r in recs if r["status"] == "answered"]
        cites = [r for r in ok if r["cites_expected"] is not None]
        web = [r for r in ok if r["category"] in SECTION_HOSTS]
        traps = [r for r in ok if r["declined"] is not None]
        times = [r["latency_s"] for r in ok]
        table[name] = {
            "model": ok[-1]["model"] if ok else (MODELS[name].model or ""),
            "asked": len(questions), "answered": len(ok),
            "cites": sum(1 for r in cites if r["cites_expected"]), "cites_of": len(cites),
            "links": sum(1 for r in web if r["has_link"]), "links_of": len(web),
            "declined": sum(1 for r in traps if r["declined"]), "declined_of": len(traps),
            "median_s": round(statistics.median(times), 1) if times else None,
            "words": round(statistics.mean(r["words"] for r in ok)) if ok else None,
            "not_answered": dict(Counter(r["status"] for r in recs if r["status"] != "answered")),
            "by_category": {c: [sum(1 for r in cites if r["category"] == c and r["cites_expected"]),
                                sum(1 for r in cites if r["category"] == c)]
                            for c in dict.fromkeys(q["category"] for q in questions) if c != "trap"},
        }
    return table


def same_context(saved, questions):
    """Did both models see the same retrieved chunks? They should, every time."""
    both = [q["id"] for q in questions if all((name, q["id"]) in saved for name in MODELS)]
    same = sum(1 for qid in both
               if saved[("gemini", qid)]["context_ids"] == saved[("ollama", qid)]["context_ids"])
    return same, len(both)


def auto_table(auto):
    measures = {
        "Model": lambda t: t["model"] or "-",
        "Answered": lambda t: pct(t["answered"], t["asked"]),
        "Cites the right source": lambda t: pct(t["cites"], t["cites_of"]),
        "Website answers with a link": lambda t: pct(t["links"], t["links_of"]),
        "Trap questions declined": lambda t: pct(t["declined"], t["declined_of"]),
        "Seconds per answer (median)": lambda t: "-" if t["median_s"] is None else t["median_s"],
        "Words per answer (average)": lambda t: "-" if t["words"] is None else t["words"],
        "Not answered": lambda t: ", ".join(f"{n} {k}" for k, n in t["not_answered"].items()) or "none",
    }
    return {m: {LABELS[name]: fn(auto[name]) for name in MODELS} for m, fn in measures.items()}


import pandas as pd
from IPython.display import Markdown, display

SAVED = load_answers()
AUTO = auto_scores(SAVED, QUESTIONS)
display(pd.DataFrame(auto_table(AUTO)).T)

print("Cites the right source, by category:")
display(pd.DataFrame({LABELS[name]: {c: pct(*AUTO[name]["by_category"][c]) for c in AUTO[name]["by_category"]}
                      for name in MODELS}))

same, both = same_context(SAVED, QUESTIONS)
print(f"Both models saw the same retrieved chunks on {same} of {both} questions"
      + ("" if same == both else " - re-crawling between runs changes the data; tick START_OVER to redo"))

,Gemini,Ollama
Model,gemini-3.8-flash,gemma4:12b
Answered,50/50 (100%),50/50 (100%)
Cites the right source,44/44 (100%),44/44 (100%)
Website answers with a link,24/24 (100%),24/24 (100%)
Trap questions declined,3/6 (50%),3/6 (50%)
Seconds per answer (median),1.5,4.4
Words per answer (average),96,101
Not answered,none,none


Cites the right source, by category:


,Gemini,Ollama
course code,10/10 (100%),10/10 (100%)
course topic,10/10 (100%),10/10 (100%)
billing,8/8 (100%),8/8 (100%)
housing,8/8 (100%),8/8 (100%)
dining,8/8 (100%),8/8 (100%)


Both models saw the same retrieved chunks on 50 of 50 questions


---
## 5 · Blind grading (recommended)

The automatic checks see whether an answer names the right course or links the right site, not whether what
it says is true. For the real verdict, grade the answers yourself.

Each question shows two answers, **A** and **B**. Which model wrote which is decided at random for each
question and stays hidden while you grade, so you judge the answer rather than the brand. Bold and heading
marks are removed in the form, because the two models format differently and that would give them away. Grade each one
*correct*, *partial* or *wrong* and press **Save grades**. You can grade in several sittings; saved grades
come back next time.

* Only questions both models answered are shown, so every comparison is like for like.
* For trap questions, *correct* means it declined or pointed to the right page without inventing anything.
* Grades are saved to `experiments/compare/grades.csv`. That file names the models, so open it only after
  you finish grading.

In [6]:
import html as html_lib


def blind_order(qid):
    """Which model is A for this question - random, but fixed so it never shuffles between sittings."""
    return ("gemini", "ollama") if random.Random(f"{BLIND_SEED}-{qid}").random() < 0.5 else ("ollama", "gemini")


def plain(text):
    """Answer text without bold or heading marks, so formatting style does not reveal the model."""
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text, flags=re.S)
    return re.sub(r"(?m)^#{1,6}\s*", "", text)


def load_grades():
    grades = {}
    if GRADES_PATH.exists():
        with open(GRADES_PATH, encoding="utf-8-sig", newline="") as f:
            for row in csv.DictReader(f):
                for name in MODELS:
                    grade = normalize_grade(row.get(f"grade_{name}"))
                    if grade:
                        grades[(name, row["id"].strip().upper())] = grade
    return grades


def save_grades(grades):
    COMPARE_DIR.mkdir(parents=True, exist_ok=True)
    with open(GRADES_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "question"] + [f"grade_{name}" for name in MODELS])
        for q in ARM1_QUESTIONS:
            writer.writerow([q["id"], q["question"]] + [grades.get((name, q["id"]), "") for name in MODELS])


def grading_form(saved, questions):
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Widgets are not available here - fill in the grade columns of grades.csv instead.")
        return
    pairs = [q for q in questions
             if all(saved.get((name, q["id"]), {}).get("status") == "answered" for name in MODELS)]
    if not pairs:
        print("Nothing to grade yet - run section 3 until both models have answered some questions.")
        return

    grades = load_grades()
    choices = [("not graded", ""), ("correct", "correct"), ("partial", "partial"), ("wrong", "wrong")]
    pickers, blocks = {}, []
    for q in pairs:
        head = (f"<b>{q['id']}</b> &middot; {html_lib.escape(q['category'])} &mdash; "
                f"{html_lib.escape(q['question'])}<br><span style='color:#666;font-size:12px'>"
                f"expected: {html_lib.escape(describe_rule(q))}</span>")
        sides = []
        for letter, name in zip("AB", blind_order(q["id"])):
            body = ("<div style='white-space:pre-wrap;border-left:3px solid #CC0000;padding:4px 10px;"
                    "margin:4px 0;max-height:260px;overflow:auto;font-size:13px'>"
                    f"{html_lib.escape(plain(saved[(name, q['id'])]['answer'])[:2500])}</div>")
            picker = widgets.Dropdown(options=choices, value=grades.get((name, q["id"]), ""),
                                      description=f"grade {letter}:")
            pickers[(name, q["id"])] = picker
            sides.append(widgets.VBox([widgets.HTML(f"<b>Answer {letter}</b>{body}"), picker],
                                      layout=widgets.Layout(width="50%", padding="0 8px 0 0")))
        blocks.append(widgets.VBox([widgets.HTML(head), widgets.HBox(sides)],
                                   layout=widgets.Layout(margin="0 0 18px 0")))

    save = widgets.Button(description="Save grades", button_style="danger", icon="check")
    out = widgets.Output()

    def on_save(_):
        for key, picker in pickers.items():
            if picker.value:
                grades[key] = picker.value
            else:
                grades.pop(key, None)
        save_grades(grades)
        out.clear_output()
        with out:
            both = sum(1 for q in pairs if all((name, q["id"]) in grades for name in MODELS))
            print(f"saved - {both} of {len(pairs)} questions graded for both answers. Run section 6 for the verdict.")

    save.on_click(on_save)
    display(widgets.HTML(f"<b>{len(pairs)} questions to grade.</b> Press <b>Save grades</b> as often as you like."))
    display(widgets.VBox(blocks + [save, out]))


grading_form(load_answers(), QUESTIONS)

HTML(value='<b>50 questions to grade.</b> Press <b>Save grades</b> as often as you like.')

---
## 6 · Verdict

Each graded answer scores 1 for *correct*, ½ for *partial* and 0 for *wrong*. A model's **graded score** is
its average over the questions graded for both models, and **head to head** counts which model got the better
grade on each one. With no grades yet, the verdict falls back to the automatic checks and says so.

Everything is saved to `experiments/compare/`: `results.csv` (one row per question, both answers side by
side), `summary.json`, and `comparison.md`, a write-up of this run.

In [7]:
SCORE = {"correct": 1.0, "partial": 0.5, "wrong": 0.0}


def manual_scores(grades, questions):
    ids = [q["id"] for q in questions if all((name, q["id"]) in grades for name in MODELS)]
    category = {q["id"]: q["category"] for q in questions}
    models = {}
    for name in MODELS:
        counts = Counter(grades[(name, qid)] for qid in ids)
        models[name] = {g: counts[g] for g in SCORE}
        models[name]["score"] = round(sum(SCORE[grades[(name, qid)]] for qid in ids) / len(ids), 3) if ids else None
    wins = Counter()
    for qid in ids:
        g, o = SCORE[grades[("gemini", qid)]], SCORE[grades[("ollama", qid)]]
        wins["gemini" if g > o else "ollama" if o > g else "tie"] += 1
    by_category = {}
    for c in dict.fromkeys(category[qid] for qid in ids):
        in_c = [qid for qid in ids if category[qid] == c]
        by_category[c] = {name: round(sum(SCORE[grades[(name, qid)]] for qid in in_c) / len(in_c), 3)
                          for name in MODELS} | {"n": len(in_c)}
    return {"graded": len(ids), "models": models, "head_to_head": {k: wins[k] for k in ("gemini", "ollama", "tie")},
            "by_category": by_category}


def verdict_lines(auto, manual):
    g_auto, o_auto = auto["gemini"], auto["ollama"]
    lines = []
    if manual["graded"]:
        g, o = manual["models"]["gemini"]["score"], manual["models"]["ollama"]["score"]
        h = manual["head_to_head"]
        if abs(g - o) < 0.05:
            lines.append(f"**About even on answer quality**: graded score Gemini {g:.0%}, Ollama {o:.0%} "
                         f"over {manual['graded']} questions.")
        else:
            lines.append(f"**{'Gemini' if g > o else 'Ollama'} gave better answers**: graded score Gemini {g:.0%}, "
                         f"Ollama {o:.0%} over {manual['graded']} questions.")
        lines.append(f"Head to head, Gemini was better on {h['gemini']}, Ollama on {h['ollama']}, "
                     f"and they tied on {h['tie']}.")
        if manual["graded"] < 20:
            lines.append("*Fewer than 20 questions are graded for both models, so treat this as a first look.*")
    else:
        def rate(t):
            num, den = t["cites"] + t["declined"], t["cites_of"] + t["declined_of"]
            return num / den if den else None
        g, o = rate(g_auto), rate(o_auto)
        lines.append("**Provisional - no grades yet.** This verdict uses only the automatic checks, which "
                     "catch missing sources but not wrong facts. Grade the answers in section 5 for a real one.")
        if g is not None and o is not None:
            if abs(g - o) < 0.05:
                lines.append(f"On the automatic checks they are about even (Gemini {g:.0%}, Ollama {o:.0%}).")
            else:
                lines.append(f"On the automatic checks {'Gemini' if g > o else 'Ollama'} leads "
                             f"(Gemini {g:.0%}, Ollama {o:.0%} of source and trap checks passed).")
    if g_auto["median_s"] is not None and o_auto["median_s"] is not None:
        lines.append(f"Speed: a median {g_auto['median_s']} s per answer for Gemini and {o_auto['median_s']} s "
                     "for Ollama.")
    lines.append(f"Reliability: Gemini answered {pct(g_auto['answered'], g_auto['asked'])} of the questions and "
                 f"Ollama {pct(o_auto['answered'], o_auto['asked'])}.")
    lines.append("Cost: Ollama runs on this computer, free and with no limit; Gemini's free tier has per-minute "
                 "and daily limits.")
    return lines


def comparison_markdown(auto, manual, same, both):
    out = ["## Gemini vs Ollama on the ISU Campus Assistant", "",
           f"*Run {datetime.now():%Y-%m-%d at %H:%M}. Gemini: {auto['gemini']['model'] or '-'}. "
           f"Ollama: {auto['ollama']['model'] or '-'}. {auto['gemini']['asked']} questions; both models saw "
           f"the same retrieved chunks on {same} of {both}.*", "", "### Verdict"]
    out += [f"- {line}" for line in verdict_lines(auto, manual)]
    out += ["", "### Automatic checks", "", "| Measure | Gemini | Ollama |", "|---|---|---|"]
    for measure, row in auto_table(auto).items():
        out.append(f"| {measure} | {row['Gemini']} | {row['Ollama']} |")
    if manual["graded"]:
        m = manual["models"]
        out += ["", f"### Blind grading ({manual['graded']} questions graded for both)", "",
                "| | Gemini | Ollama |", "|---|---|---|"]
        for g in SCORE:
            out.append(f"| {g.title()} | {m['gemini'][g]} | {m['ollama'][g]} |")
        out.append(f"| **Graded score** | **{m['gemini']['score']:.0%}** | **{m['ollama']['score']:.0%}** |")
        out += ["", "| Category | Questions | Gemini | Ollama |", "|---|---|---|---|"]
        for c, row in manual["by_category"].items():
            out.append(f"| {c} | {row['n']} | {row['gemini']:.0%} | {row['ollama']:.0%} |")
    else:
        out += ["", "*No blind grades yet - grade the answers in section 5 and run this cell again.*"]
    return "\n".join(out)


def save_comparison(saved, grades, auto, manual, report):
    COMPARE_DIR.mkdir(parents=True, exist_ok=True)
    fields = ["id", "category", "question", "rule"]
    per_model = ["status", "latency_s", "cites_expected", "declined", "has_link", "words", "grade", "answer"]
    with open(COMPARE_DIR / "results.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(fields + [f"{name}_{col}" for name in MODELS for col in per_model])
        for q in QUESTIONS:
            row = [q["id"], q["category"], q["question"], describe_rule(q)]
            for name in MODELS:
                rec = saved.get((name, q["id"]), {})
                row += [rec.get(col, "") if rec.get(col) is not None else "" for col in per_model[:-2]]
                row += [grades.get((name, q["id"]), ""), rec.get("answer", "")]
            writer.writerow(row)
    (COMPARE_DIR / "summary.json").write_text(json.dumps({"automatic": auto, "manual": manual}, indent=2),
                                              encoding="utf-8")
    (COMPARE_DIR / "comparison.md").write_text(report, encoding="utf-8")
    print(f"saved to {COMPARE_DIR}: results.csv, summary.json, comparison.md")


SAVED, GRADES = load_answers(), load_grades()
AUTO = auto_scores(SAVED, QUESTIONS)
MANUAL = manual_scores(GRADES, QUESTIONS)
REPORT = comparison_markdown(AUTO, MANUAL, *same_context(SAVED, QUESTIONS))
save_comparison(SAVED, GRADES, AUTO, MANUAL, REPORT)
display(Markdown(REPORT))

saved to isu_catalog_cache\experiments\compare: results.csv, summary.json, comparison.md


## Gemini vs Ollama on the ISU Campus Assistant

*Run 2026-10-16 at 10:34. Gemini: gemini-3.8-flash. Ollama: gemma4:12b. 50 questions; both models saw the same retrieved chunks on 50 of 50.*

### Verdict
- **Provisional - no grades yet.** This verdict uses only the automatic checks, which catch missing sources but not wrong facts. Grade the answers in section 5 for a real one.
- On the automatic checks they are about even (Gemini 94%, Ollama 94%).
- Speed: a median 1.5 s per answer for Gemini and 4.4 s for Ollama.
- Reliability: Gemini answered 50/50 (100%) of the questions and Ollama 50/50 (100%).
- Cost: Ollama runs on this computer, free and with no limit; Gemini's free tier has per-minute and daily limits.

### Automatic checks

| Measure | Gemini | Ollama |
|---|---|---|
| Model | gemini-3.8-flash | gemma4:12b |
| Answered | 50/50 (100%) | 50/50 (100%) |
| Cites the right source | 44/44 (100%) | 44/44 (100%) |
| Website answers with a link | 24/24 (100%) | 24/24 (100%) |
| Trap questions declined | 3/6 (50%) | 3/6 (50%) |
| Seconds per answer (median) | 1.5 | 4.4 |
| Words per answer (average) | 96 | 101 |
| Not answered | none | none |

*No blind grades yet - grade the answers in section 5 and run this cell again.*